# Aula 26 — KD-Trees no Projeto Aqua Index

**UNIFEI • ECAA08 Automática • Grupo 2**  
**Projeto:** Aqua Index — SCADA da Máquina Pneumática de Envasamento de Água em Copos

> **Nota de Escopo Conceitual:** Implementação didática em **Python 3.13 + NumPy** (sem `scipy` ou `sklearn`; `matplotlib` opcional em `try/except`). Não representa máquina física construída, não adiciona sensores físicos novos ao CLP e não reivindica certificação de segurança funcional.

In [1]:
import heapq
import math
import time
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

SEED = 2026
rng = np.random.default_rng(SEED)
print(f"Python/NumPy pronto | NumPy {np.__version__} | matplotlib={HAS_MATPLOTLIB}")

Python/NumPy pronto | NumPy 2.5.4 | matplotlib=False

In [2]:
class KDNode:
    """Nó de uma KD-Tree para espaços contínuos R^k."""
    __slots__ = ("point", "index", "axis", "left", "right")

    def __init__(self, point, index, axis, left=None, right=None):
        self.point = point
        self.index = index
        self.axis = axis
        self.left = left
        self.right = right


class KDTree:
    """Implementação do zero de KD-Tree com k-NN e Orthogonal Range Search."""

    def __init__(self, points):
        pts = np.asarray(points, dtype=np.float64)
        if pts.ndim != 2 or pts.shape[0] == 0:
            raise ValueError("points deve ser uma matriz 2D não vazia (N x k).")
        self.points = pts
        self.n_samples, self.k = pts.shape
        indices = np.arange(self.n_samples, dtype=np.int64)
        self.root = self._build(indices, depth=0)

    def _build(self, indices, depth):
        if len(indices) == 0:
            return None
        axis = depth % self.k
        order = indices[np.argsort(self.points[indices, axis], kind="mergesort")]
        mid = len(order) // 2
        idx = int(order[mid])
        return KDNode(
            point=self.points[idx],
            index=idx,
            axis=axis,
            left=self._build(order[:mid], depth + 1),
            right=self._build(order[mid + 1 :], depth + 1),
        )

    def k_nearest(self, query, k=1):
        """Retorna lista ordenada de (distancia_euclidiana, indice) e nós visitados."""
        if k < 1 or k > self.n_samples:
            raise ValueError("k deve estar entre 1 e n_samples.")
        q = np.asarray(query, dtype=np.float64)
        heap = []  # max-heap via (-dist_sq, -index, index)
        visited = 0

        def _search(node):
            nonlocal visited
            if node is None:
                return
            visited += 1
            diff_vec = q - node.point
            dist_sq = float(np.dot(diff_vec, diff_vec))

            if len(heap) < k:
                heapq.heappush(heap, (-dist_sq, -node.index, node.index))
            elif dist_sq < -heap[0][0] or (dist_sq == -heap[0][0] and node.index < heap[0][2]):
                heapq.heapreplace(heap, (-dist_sq, -node.index, node.index))

            axis = node.axis
            delta = float(q[axis] - node.point[axis])
            near_branch = node.left if delta <= 0.0 else node.right
            far_branch = node.right if delta <= 0.0 else node.left

            _search(near_branch)

            if len(heap) < k or (delta * delta) <= -heap[0][0]:
                _search(far_branch)

        _search(self.root)
        result = sorted((math.sqrt(-neg_d2), idx) for neg_d2, _, idx in heap)
        return result, visited

    def nearest(self, query):
        """Atalho para 1-NN: retorna (distancia, indice, nos_visitados)."""
        res, visited = self.k_nearest(query, k=1)
        dist, idx = res[0]
        return dist, idx, visited

    def range_search(self, low, high):
        """Orthogonal Range Search no hiper-retângulo fechado [low, high]."""
        lo = np.asarray(low, dtype=np.float64)
        hi = np.asarray(high, dtype=np.float64)
        if lo.shape != (self.k,) or hi.shape != (self.k,):
            raise ValueError("low e high devem ter dimensão k.")
        matches = []
        visited = 0

        def _range(node):
            nonlocal visited
            if node is None:
                return
            visited += 1
            p = node.point
            if np.all((p >= lo) & (p <= hi)):
                matches.append(node.index)

            axis = node.axis
            if lo[axis] <= p[axis]:
                _range(node.left)
            if hi[axis] >= p[axis]:
                _range(node.right)

        _range(self.root)
        matches.sort()
        return matches, visited


def brute_k_nearest(points, query, k=1):
    q = np.asarray(query, dtype=np.float64)
    diffs = points - q
    dists_sq = np.einsum("ij,ij->i", diffs, diffs)
    order = np.argsort(dists_sq, kind="mergesort")[:k]
    return [(float(math.sqrt(dists_sq[i])), int(i)) for i in order]


def brute_range_search(points, low, high):
    lo = np.asarray(low, dtype=np.float64)
    hi = np.asarray(high, dtype=np.float64)
    mask = np.all((points >= lo) & (points <= hi), axis=1)
    return [int(i) for i in np.flatnonzero(mask)]

In [3]:
# Validação 100% contra força bruta em R^2 e R^3 (incluindo bordas e duplicatas)
for dim in (2, 3):
    pts_test = rng.uniform(-10.0, 10.0, size=(300, dim))
    # Injeta pontos duplicados e exatamente na borda para testar desempate
    pts_test[0] = np.zeros(dim)
    pts_test[1] = np.zeros(dim)
    tree_test = KDTree(pts_test)

    for _ in range(40):
        q = rng.uniform(-12.0, 12.0, size=dim)
        for k_val in (1, 5):
            kd_res, _ = tree_test.k_nearest(q, k=k_val)
            bf_res = brute_k_nearest(pts_test, q, k=k_val)
            assert len(kd_res) == len(bf_res) == k_val
            for (d_kd, i_kd), (d_bf, i_bf) in zip(kd_res, bf_res):
                assert abs(d_kd - d_bf) < 1e-12, f"Divergência distância: {d_kd} vs {d_bf}"
                assert i_kd == i_bf, f"Divergência índice: {i_kd} vs {i_bf}"

        c1 = rng.uniform(-8.0, 4.0, size=dim)
        c2 = c1 + rng.uniform(0.5, 8.0, size=dim)
        kd_rng, _ = tree_test.range_search(c1, c2)
        bf_rng = brute_range_search(pts_test, c1, c2)
        assert kd_rng == bf_rng, f"Divergência range_search em R^{dim}"

    # Caso exato na origem duplicada
    kd_zero, _ = tree_test.k_nearest(np.zeros(dim), k=2)
    assert kd_zero[0][0] == 0.0 and kd_zero[1][0] == 0.0

print("VALIDAÇÃO 100% APROVADA: k-NN (k=1, k=5) e range_search idênticos à força bruta em R^2 e R^3.")

VALIDAÇÃO 100% APROVADA: k-NN (k=1, k=5) e range_search idênticos à força bruta em R^2 e R^3.

In [4]:
# ==============================================================================
# FRENTE A — Historiador Multidimensional SCADA em R^3 (Complemento à AVL 1D)
# Vetor contínuo sem novos sensores físicos:
#   [TIT-401 (°C), takt_s (derivado de SE-001), tempo_vacuo_ms (derivado VAC-301 -> PIT-301)]
# ==============================================================================

N_CICLOS = 2400
# 4 regimes operacionais realistas (sem inventar sensores)
n_nom = 1800   # Operação nominal
n_frio = 220   # Selagem fria / aquecimento (R-02 hold)
n_vac = 230    # Vácuo lento / filtro sujo em E3 (aprox. R-03 retry)
n_misto = 150  # Degradação combinada (frio + vácuo lento + takt alto)

tit_nom = rng.normal(185.0, 2.0, n_nom)
takt_nom = rng.normal(3.60, 0.16, n_nom)
vac_nom = rng.normal(215.0, 20.0, n_nom)

tit_frio = rng.normal(175.5, 2.2, n_frio)
takt_frio = rng.normal(4.55, 0.25, n_frio)
vac_frio = rng.normal(225.0, 22.0, n_frio)

tit_vac = rng.normal(184.5, 2.1, n_vac)
takt_vac = rng.normal(4.35, 0.22, n_vac)
vac_vac = rng.normal(435.0, 40.0, n_vac)

tit_misto = rng.normal(177.0, 2.0, n_misto)
takt_misto = rng.normal(5.10, 0.30, n_misto)
vac_misto = rng.normal(460.0, 45.0, n_misto)

raw_scada = np.vstack([
    np.column_stack([tit_nom, takt_nom, vac_nom]),
    np.column_stack([tit_frio, takt_frio, vac_frio]),
    np.column_stack([tit_vac, takt_vac, vac_vac]),
    np.column_stack([tit_misto, takt_misto, vac_misto]),
])
labels_scada = (
    ["NOMINAL"] * n_nom
    + ["SELAGEM_FRIA_R02"] * n_frio
    + ["VACUO_LENTO_R03"] * n_vac
    + ["DEGRADACAO_MISTA"] * n_misto
)

# Embaralha ordem temporal mantendo reprodutibilidade
perm = rng.permutation(N_CICLOS)
raw_scada = raw_scada[perm]
labels_scada = [labels_scada[i] for i in perm]

# Normalização Z-score obrigatória (evita dominância de tempo_vacuo_ms sobre takt_s)
mu = raw_scada.mean(axis=0)
sigma = raw_scada.std(axis=0)
norm_scada = (raw_scada - mu) / sigma

kd_scada = KDTree(norm_scada)

# 1) Orthogonal Range Search: janela de operação nominal
low_raw = np.array([180.0, 3.20, 150.0])
high_raw = np.array([190.0, 4.00, 280.0])
low_z = (low_raw - mu) / sigma
high_z = (high_raw - mu) / sigma

kd_range_idx, range_visited = kd_scada.range_search(low_z, high_z)
bf_range_idx = brute_range_search(norm_scada, low_z, high_z)
assert kd_range_idx == bf_range_idx
assert len(kd_range_idx) > 0

# 2) k-NN (k=5) para diagnóstico de ciclo anômalo
query_anomalo_raw = np.array([177.5, 4.70, 445.0])
query_anomalo_z = (query_anomalo_raw - mu) / sigma
kd_knn, knn_visited = kd_scada.k_nearest(query_anomalo_z, k=5)
bf_knn = brute_k_nearest(norm_scada, query_anomalo_z, k=5)

assert [i for _, i in kd_knn] == [i for _, i in bf_knn]
for (d1, _), (d2, _) in zip(kd_knn, bf_knn):
    assert abs(d1 - d2) < 1e-12

print("=== FRENTE A: Historiador Multidimensional SCADA (R^3) ===")
print(f"Total de ciclos indexados: {N_CICLOS}")
print(f"Médias [TIT-401, takt_s, vacuo_ms]: {np.round(mu, 2)}")
print(f"Desvios-padrão: {np.round(sigma, 2)}")
print(
    f"Orthogonal Range Search [180..190°C, 3.2..4.0s, 150..280ms]: "
    f"{len(kd_range_idx)} ciclos | nós visitados = {range_visited}/{N_CICLOS} "
    f"({100.0 * range_visited / N_CICLOS:.1f}%) | 100% concordância força bruta"
)
print(
    f"5-NN para ciclo anômalo {query_anomalo_raw.tolist()}: "
    f"nós visitados = {knn_visited}/{N_CICLOS} ({100.0 * knn_visited / N_CICLOS:.1f}%)"
)
for rank, (dist_z, idx) in enumerate(kd_knn, start=1):
    print(
        f"  #{rank}: ciclo={idx:4d} | dist_z={dist_z:.4f} | "
        f"raw={np.round(raw_scada[idx], 2).tolist()} | regime={labels_scada[idx]}"
    )

=== FRENTE A: Historiador Multidimensional SCADA (R^3) ===
Total de ciclos indexados: 2400
Médias [TIT-401, takt_s, vacuo_ms]: [183.56   3.86 251.79]
Desvios-padrão: [ 3.83  0.5  86.47]
Orthogonal Range Search [180..190°C, 3.2..4.0s, 150..280ms]: 1751 ciclos | nós visitados = 1840/2400 (76.7%) | 100% concordância força bruta
5-NN para ciclo anômalo [177.5, 4.7, 445.0]: nós visitados = 54/2400 (2.2%)
  #1: ciclo=1362 | dist_z=0.2596 | raw=[177.8, 4.82, 441.63] | regime=DEGRADACAO_MISTA
  #2: ciclo=1296 | dist_z=0.3282 | raw=[177.43, 4.86, 447.65] | regime=DEGRADACAO_MISTA
  #3: ciclo=1723 | dist_z=0.3504 | raw=[178.82, 4.69, 440.32] | regime=DEGRADACAO_MISTA
  #4: ciclo= 134 | dist_z=0.3768 | raw=[177.39, 4.87, 460.06] | regime=DEGRADACAO_MISTA
  #5: ciclo=1891 | dist_z=0.4322 | raw=[176.41, 4.85, 454.97] | regime=DEGRADACAO_MISTA

In [5]:
# ==============================================================================
# FRENTE B — Aceleração do passo NEAREST do RRT (Aula 22) em R^2 (16 m x 12 m)
# Benchmark O(N) linear vs KD-Tree O(log N) para N in {200, 1000, 5000, 15000}
# ==============================================================================

def gerar_vertices_rrt_galpao(n_vertices, rng_obj):
    """Gera vértices no espaço livre do galpão 16 m x 12 m (Aula 22)."""
    pts = []
    # Raiz em ESTOQUE_COPOS (1.60, 1.80)
    pts.append([1.60, 1.80])
    while len(pts) < n_vertices:
        cand = rng_obj.uniform([0.45, 0.45], [15.55, 11.55], size=(n_vertices, 2))
        # Exclui disco da célula MESA-001 (8.0, 6.2), raio efetivo 2.20 m
        dx = cand[:, 0] - 8.0
        dy = cand[:, 1] - 6.2
        livre = (dx * dx + dy * dy) > (2.20 ** 2)
        for p in cand[livre]:
            pts.append([float(p[0]), float(p[1])])
            if len(pts) == n_vertices:
                break
    return np.asarray(pts, dtype=np.float64)


def linear_nearest(points, query):
    diffs = points - query
    dists_sq = np.einsum("ij,ij->i", diffs, diffs)
    idx = int(np.argmin(dists_sq))
    return float(math.sqrt(dists_sq[idx])), idx, len(points)


N_LIST = [200, 1000, 5000, 15000]
N_QUERIES = 200
bench_results = []

print("=== FRENTE B: Benchmark NEAREST no RRT (R^2, galpão 16 m x 12 m) ===")
print(f"{'N':>7} | {'Linear (ms)':>11} | {'KD-Tree (ms)':>12} | {'Speedup':>8} | {'Nós KD méd.':>12} | {'Redução nós':>11}")
print("-" * 76)

for n_pts in N_LIST:
    verts = gerar_vertices_rrt_galpao(n_pts, rng)
    queries = rng.uniform([0.45, 0.45], [15.55, 11.55], size=(N_QUERIES, 2))
    # Última consulta próxima da doca MAG-100 (13.20, 9.60)
    queries[-1] = np.array([13.20, 9.60])

    kd_rrt = KDTree(verts)

    # Linear scan
    t0 = time.perf_counter()
    lin_res = [linear_nearest(verts, q) for q in queries]
    t_lin_ms = (time.perf_counter() - t0) * 1000.0

    # KD-Tree
    t0 = time.perf_counter()
    kd_res = [kd_rrt.nearest(q) for q in queries]
    t_kd_ms = (time.perf_counter() - t0) * 1000.0

    # Validação de concordância 100%
    visited_counts = []
    for (d_l, i_l, _), (d_k, i_k, vis) in zip(lin_res, kd_res):
        assert abs(d_l - d_k) < 1e-12, f"Distância divergente em N={n_pts}"
        assert i_l == i_k, f"Índice divergente em N={n_pts}: {i_l} vs {i_k}"
        visited_counts.append(vis)

    avg_vis = float(np.mean(visited_counts))
    reducao_pct = 100.0 * (1.0 - avg_vis / n_pts)
    speedup = t_lin_ms / t_kd_ms if t_kd_ms > 0 else float("inf")
    bench_results.append((n_pts, t_lin_ms, t_kd_ms, speedup, avg_vis, reducao_pct))

    print(
        f"{n_pts:7d} | {t_lin_ms:11.3f} | {t_kd_ms:12.3f} | {speedup:7.2f}x | "
        f"{avg_vis:12.1f} | {reducao_pct:10.2f}%"
    )

# Asserts de eficiência para N grandes
assert bench_results[-1][4] < 0.02 * bench_results[-1][0], "KD-Tree em N=15000 deveria visitar < 2% dos nós"
assert bench_results[-1][3] > 1.0, "KD-Tree deve ser mais rápida que busca linear em N=15000"
print("VALIDAÇÃO FRENTE B APROVADA: 100% vizinho idêntico e redução expressiva de nós visitados.")

=== FRENTE B: Benchmark NEAREST no RRT (R^2, galpão 16 m x 12 m) ===
      N | Linear (ms) | KD-Tree (ms) |  Speedup |  Nós KD méd. | Redução nós
----------------------------------------------------------------------------
    200 |       2.607 |        8.160 |    0.32x |         13.4 |      93.30%
   1000 |       5.026 |        8.727 |    0.58x |         16.8 |      98.32%
   5000 |      11.256 |        8.443 |    1.33x |         22.4 |      99.55%
  15000 |      27.648 |       10.455 |    2.64x |         23.6 |      99.84%
VALIDAÇÃO FRENTE B APROVADA: 100% vizinho idêntico e redução expressiva de nós visitados.

In [6]:
# ==============================================================================
# MALDIÇÃO DA DIMENSIONALIDADE: Eficiência de poda em d = 2, 3 vs d >= 20
# ==============================================================================

DIMS = [2, 3, 5, 10, 20, 30, 50]
N_CURSE = 2000
Q_CURSE = 100

print("=== Maldição da Dimensionalidade (N = 2000, 100 consultas 1-NN) ===")
print(f"{'Dim (d)':>7} | {'Nós visitados méd.':>18} | {'% da árvore':>12}")
print("-" * 44)

curse_ratios = {}
for d in DIMS:
    pts_d = rng.uniform(0.0, 1.0, size=(N_CURSE, d))
    queries_d = rng.uniform(0.0, 1.0, size=(Q_CURSE, d))
    kd_d = KDTree(pts_d)
    vis_list = []
    for q in queries_d:
        _, _, vis = kd_d.nearest(q)
        vis_list.append(vis)
    avg_v = float(np.mean(vis_list))
    pct = 100.0 * avg_v / N_CURSE
    curse_ratios[d] = pct
    print(f"{d:7d} | {avg_v:18.1f} | {pct:11.2f}%")

assert curse_ratios[2] < 5.0, "Em R^2 a poda deve visitar < 5% dos nós"
assert curse_ratios[3] < 10.0, "Em R^3 a poda deve visitar < 10% dos nós"
assert curse_ratios[20] > 75.0, "Para d >= 20 a poda colapsa (> 75% dos nós visitados)"
assert curse_ratios[30] > 95.0, "Para d = 30 praticamente toda a árvore é visitada"
print("VALIDAÇÃO DIMENSIONALIDADE APROVADA: poda eficiente em R^2/R^3 e colapso em d >= 20.")

=== Maldição da Dimensionalidade (N = 2000, 100 consultas 1-NN) ===
Dim (d) | Nós visitados méd. |  % da árvore
--------------------------------------------
      2 |               16.7 |        0.84%
      3 |               28.2 |        1.41%
      5 |               99.0 |        4.95%
     10 |             1299.0 |       64.95%
     20 |             1999.1 |       99.96%
     30 |             2000.0 |      100.00%
     50 |             2000.0 |      100.00%
VALIDAÇÃO DIMENSIONALIDADE APROVADA: poda eficiente em R^2/R^3 e colapso em d >= 20.

In [7]:
# Visualização opcional (protegida por try/except conforme requisitos)
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    ns = [r[0] for r in bench_results]
    t_lin = [r[1] for r in bench_results]
    t_kd = [r[2] for r in bench_results]
    axes[0].plot(ns, t_lin, "o-", label="Linear O(N)")
    axes[0].plot(ns, t_kd, "s-", label="KD-Tree O(log N)")
    axes[0].set_xlabel("N vértices RRT")
    axes[0].set_ylabel("Tempo total 200 queries (ms)")
    axes[0].set_title("Frente B: Benchmark NEAREST em R^2")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    ds = list(curse_ratios.keys())
    pcts = [curse_ratios[d] for d in ds]
    axes[1].plot(ds, pcts, "o-", color="crimson")
    axes[1].set_xlabel("Dimensão d")
    axes[1].set_ylabel("% de nós visitados")
    axes[1].set_title("Maldição da Dimensionalidade")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib não disponível neste ambiente; validação numérica concluída sem gráficos.")

matplotlib não disponível neste ambiente; validação numérica concluída sem gráficos.